In [63]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://books.toscrape.com"

response = requests.get(url)
response.raise_for_status() # check if the request was successful

print(response.status_code)

print(response.text[:1000])

200
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="static/oscar/favi

In [64]:
soup = BeautifulSoup(response.text, 'html.parser')

print(soup.title)

books = soup.find_all('article', class_ = 'product_pod')

<title>
    All products | Books to Scrape - Sandbox
</title>


In [65]:
print(soup.title.text)
print(len(books))


    All products | Books to Scrape - Sandbox

20


In [66]:
book_soup = BeautifulSoup(response.text, 'html.parser')

print(book_soup.title.text)



    All products | Books to Scrape - Sandbox



In [67]:
# Retrieve Book Details: Title, Price, Rating, Availability and category
books_data = []
for book in books:
    #title = book.h3.attrs['title']
    title = book.h3.a.attrs['title']
    price = book.find('p', class_ =  'price_color').text
    rating = book.p.attrs['class'][1]
    availability = book.find('p', class_ = 'instock availability').text.strip()

    # -------------------------
    # Category
    # -------------------------
    category = soup.select_one("ul.breadcrumb li:nth-of-type(2)")
    
    books_data.append({
        "title": title,
        "price": price,
        "rating": rating,
        "availability": availability,
        "category": category    
    })

df_books_data = pd.DataFrame(books_data)

print(df_books_data.head())


                                   title    price rating availability  \
0                   A Light in the Attic  Â£51.77  Three     In stock   
1                     Tipping the Velvet  Â£53.74    One     In stock   
2                             Soumission  Â£50.10    One     In stock   
3                          Sharp Objects  Â£47.82   Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23   Five     In stock   

         category  
0  [All products]  
1  [All products]  
2  [All products]  
3  [All products]  
4  [All products]  


In [68]:
#category = soup.select_one("ul.breadcrumb")
category = soup.select_one("ul.breadcrumb li:nth-of-type(1)")
#print(category)
print(category.getText(strip=True))

Home


In [69]:
# Retrieve the subcategory of books - then retrieve category_id and category_name
subcategories = soup.select("ul.nav-list ul li a")

for subcategory in subcategories:
    subcategory_name = subcategory.getText(strip=True)
    subcategory_url = url + '/' + subcategory.attrs['href']

    print(f"Subcategory: {subcategory_name}, URL: {subcategory_url}")

    category_id = int(subcategory_url.split("_")[-1].split("/")[0])

    print(f"Category ID: {category_id}, Category Name: {subcategory_name}")


Subcategory: Travel, URL: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Category ID: 2, Category Name: Travel
Subcategory: Mystery, URL: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Category ID: 3, Category Name: Mystery
Subcategory: Historical Fiction, URL: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Category ID: 4, Category Name: Historical Fiction
Subcategory: Sequential Art, URL: https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Category ID: 5, Category Name: Sequential Art
Subcategory: Classics, URL: https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Category ID: 6, Category Name: Classics
Subcategory: Philosophy, URL: https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
Category ID: 7, Category Name: Philosophy
Subcategory: Romance, URL: https://books.toscrape.com/catalogue/category/books/romance_8/index.html
Cat

In [70]:
# Convert Price: from GBP to INR

df_books_data["price"] = df_books_data["price"].str.replace("£", "", regex=False)
df_books_data["price"] = df_books_data["price"].str.replace("Â", "", regex=False)  # Remove any unwanted characters

df_books_data["price"] = df_books_data["price"].astype(float) #* 100  # Assuming 1 GBP = 100 INR

In [10]:
# Convert Rating: from string to numeric
rating_map = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}
df_books_data['rating_numeric'] = df_books_data['rating'].map(rating_map)

print(df_books_data[['title', 'price', 'rating_numeric', 'availability']].head())


                                   title  price  rating_numeric availability
0                   A Light in the Attic  51.77               3     In stock
1                     Tipping the Velvet  53.74               1     In stock
2                             Soumission  50.10               1     In stock
3                          Sharp Objects  47.82               4     In stock
4  Sapiens: A Brief History of Humankind  54.23               5     In stock


In [86]:
from urllib.parse import urljoin

# Find all products page
base_url = "https://books.toscrape.com"

# --------------------------------------------------
# STEP 1: Get the All Products page
# --------------------------------------------------
response = requests.get(base_url)
response.raise_for_status()  # check if the request was successful
soup = BeautifulSoup(response.text, "html.parser")

# --------------------------------------------------
# STEP 2: Find all subcategory links under Books
# --------------------------------------------------
category_links = {}
categories = []

# Find the Books section in the sidebar
#category_link = soup.select_one("ul.nav-list li a")
books_section = soup.select_one("div.side_categories ul.nav-list > li > ul")

# Get all subcategory links under Books
for link in books_section.find_all("a"):
    subcategory_name = link.get_text(strip=True)
    subbcategory_url = urljoin(base_url, link['href'])
    category_links[subcategory_name] = subbcategory_url

    category_id = link['href'].split("_")[-1].split("/")[0]

    categories.append({"category_id": category_id, "category_name": subcategory_name})

print(categories)

# Display available categories
print("Available Categories:")
print(category_links.keys())

# --------------------------------------------------
# STEP 3: Select 4 categories
# --------------------------------------------------
selected_categories = ["Travel", "Historical Fiction", "Mystery", 'Sequential Art']

# --------------------------------------------------
# STEP 4: Scrape books from the 4 categories
# --------------------------------------------------
all_books = []

for category in selected_categories:
    url = category_links[category]
    response = requests.get(url)
    response.raise_for_status()  # check if the request was successful

    category_soup = BeautifulSoup(response.text, "html.parser")

    # Find all books on currecnt page
    books = category_soup.find_all("article", class_="product_pod")

    '''categories.append({
        "category_id": category_id,
        "category_name": subcategory_name
    }) '''

    print(df_books.columns.to_list())
    
    for book in books:
        # Get <a> tag inside <h3>
        book_link = book.find("h3").find("a")

        # Get book ID from URL
        book_url = book_link["href"]
        book_id = int(book_url.rsplit("_", 1)[1].split("/")[0])
        #book_id = book.h3.attrs.get('book_id') 
        title = book.h3.a.attrs.get('title')
        price = book.find("p", class_='price_color').text.strip()
        rating = book.p.attrs.get('class')[1]
        availability = book.find("p", class_='instock availability').text.strip()

        all_books.append({
            "book_id": book_id,
            "category_id": category_id,
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability
        })

        print(f"book_id: {book_id}")
        print(f"title: {title}")

        # - Parse the availability text into a boolean column 
        #           in_stock. 
        #all_books['in_stock'] = book['availability'].eq('In stock')
        #all_books['in_stock'] = book['in_stock'].astype(bool)
        #all_books['in_stock'] = (
        #    all_books['availability']
        #    .str.strip()
        #    .str.lower()
        #    .eq('in stock')
        #)


        #book['in_stock'] = book['in_stock'].astype(bool)

        #print(book[['in_stock', 'in_stock']].head(20))
        #print(df_books['in_stock'].dtype)

        # Parse availability text into a boolean column: in_stock
        all_books['in_stock'] = (
        all_books['availability']
        .str.strip()
        .str.lower()
        .eq('in stock'))

        # ------------------------------------------
        # Find Next page
        # ------------------------------------------
        next_button = category_soup.select_one(
            "li.next a"
        )
        if next_button:
            next_url = next_button["href"]
            url = urljoin(
                url,
                next_url
            )
        else:
            # No more pages
            url = None
# --------------------------------------------------
# STEP 5: Create DataFrame
# --------------------------------------------------
df_books = pd.DataFrame(all_books)
df_category = pd.DataFrame(categories)

# --------------------------------------------------
# STEP 6: Display results
# --------------------------------------------------
print("\nScraping completed!")
print("Total books:", len(df_books))
print("Total categories:", len(df_category))
print("\nBooks by category:")
print(df_books["category_id"].value_counts())
print("\nFirst 60 records:")
print(df_books.head(60))

# --------------------------------------------------------------------------
# Task-2: Clean the scraped fields into proper types:
#         - Strip the currency symbol from price and convert it 
#           to a float column price_gbp. 
#         - Convert the text star - rating (One…Five) into an integer column 
#           rating (1–5).
#         - Parse the availability text into a boolean column 
#           in_stock. 
#         - If any field fails to parse for a given row 
#           (e.g., unexpected text), handle it with the median-imputation 
#           approach for numeric fields or drop the row (state and justify 
#           your choice) — do not leave the pipeline crashing on messy rows.
# ----------------------------------------------------------------------------
# Convert Price: from GBP to INR
# Remove any unwanted characters
all_books["price_gbp"] = df_books["price"]
all_books["price_inr"] = df_books["price"].str.replace("Â£", "", regex=False)

# - Parse the availability text into a boolean column 
#           in_stock. 
#all_books['in_stock'] = df_books['availability'].eq('In stock')
#all_books['in_stock'] = df_books['in_stock'].astype(bool)

##
'''
# --------------------------------------------------------------------------
# Task-3: Convert price_gbp to a price_inr column using the project's fixed 
#         baseline conversion rate: 1 GBP = 105.50 INR
# ---------------------------------------------------------------------------

all_books["price_inr"] = (all_books["price_inr"].astype(float)) * 105.50  # Assuming 1 GBP = 105.50 INR

# Convert Rating: from text to integer
# - Convert the text star - rating (One…Five) into an integer column 
#           rating (1–5).
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

all_books["rating"] = all_books["rating"].map(rating_map)

print(all_books[['title', 'price_inr', 'rating', 'availability', 'category_id']].head(60))

# ---------------------------------------------------------------------------
# Task-4: Store the cleaned data into a SQLite database with two tables:
#         - books: with columns title, price_inr, rating, in_stock, category
#         - categories: with columns category_id, category_name
# ---------------------------------------------------------------------------
# Create SQLite database connection
conn = sqlite3.connect("books_data.db")

books = all_books.to_sql("books", conn, if_exists="replace", index=False)
categories = df_category.to_sql("categories", conn, if_exists="replace", index=False)

print("Data stored in SQLite database 'books_data.db' successfully!")
print("Database tables: 'books' and 'categories' created.")
print("Total records inserted into 'books' table: ", len(df_books))
print("Total records inserted into 'categories' table: ", len(df_category))


# Parsed the availability text into a boolean column 
#           in_stock.
# Parsed the price float into a float price_gbp column 
# Hence both columns are not needed
#all_books.drop(columns='price', inplace=True)
#all_books.drop(columns='availability', inplace=True)
#all_books.head(35)

_IncompleteInputError: incomplete input (4201017288.py, line 171)

In [ ]:
#import sqlite3

#conn = sqlite3.connect("book_data.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE CATEGORIES_NEW (
        category_id INTEGER PRIMARY KEY,
        category_name TEXT
    )
""")

cursor.execute("""
    CREATE TABLE BOOKS_NEW (
        book_id INTEGER PRIMARY KEY,
        category_id INTEGER,
        title TEXT,
        rating INTEGER,
        in_stock BOOL,
        price_gbp NUMBER,
        price_inr NUMBER,
        FOREIGN KEY (category_id) REFERENCES CATEGORIES_NEW(category_id)
    )
""")

'''
cursor.execute("""
    INSERT INTO CATEGORIES_NEW
    SELECT * FROM CATEGORIES
"""
)
conn.commit()

cursor.execute("""
    DROP TABLE CATEGORIES
""")
'''

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
""")

print(cursor.fetchall())

cursor.execute(""" 
    PRAGMA table_info(CATEGORIES)
""")

columns = cursor.fetchall()

for column in columns:
    print(column)

cursor.execute(""" 
    PRAGMA table_info(BOOKS)
""")

columns = cursor.fetchall()

for column in columns:
    print(column)


[('books',), ('categories',)]
(0, 'category_id', 'TEXT', 0, None, 0)
(1, 'category_name', 'TEXT', 0, None, 0)
(0, 'category', 'TEXT', 0, None, 0)
(1, 'title', 'TEXT', 0, None, 0)
(2, 'price', 'TEXT', 0, None, 0)
(3, 'rating', 'TEXT', 0, None, 0)
(4, 'availability', 'TEXT', 0, None, 0)
